# CA2 

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
data= pd.read_csv("Agri_2025.csv", encoding="latin1")

#### DATA CLEANING AND PREPARATION

SGiven that the original dataset originates from the Dirección General del Servicio de Información Agroalimentaria y Pesquera (DGSIAP, 2025) and is in Spanish, the following preprocessing steps will be performed to prepare the data for analysis:

1. Encoding and translation:
   
   The original file will be loaded using "latin1" encoding to preserve special characters. Column headers and key categorical variables ("Irrigation_Modality", "Productive_Cycle", and "Unit") will be translated into English. Proper geographical names (such as "State" and "Municipality") will be kept in their original Spanish form to maintain the official administrative nomenclature.
3. Standaridization of units and handling missing values:
   
   The dataset will be filtered to retain only records measured in metric tons ("Ton") that show a positive harvested area ("Harvested_Area_Ha > 0"). This ensures consistency in measurements and allows for the removal of missing values ("NaN") in the yield and rural price fields—missing values, that arise due to crops that were either not harvested or were damaged.


Once these cleaning stages are complete, descriptive statistics will be examined to understand the distribution of the variables and to guide the selection of appropriate statistical models.

In [3]:
columns_tras={
    "Anio": "Year", 
    "Idestado":"State_ID",
    "Nomestado":"State",
    "Idddr": "ID_DDR",
    "Nomddr":"DDR_Name",
    "Idcader":"ID_CADER",
    "Nomcader":"Name_CADER",
    "Idmunicipio":"Municipality_ID",
    "Nommunicipio":"Municipality_Name",
    "Idciclo":"Cycle_ID",
    "Nomcicloproductivo":"Productive_Cycle",
    "Idmodalidad":"Modality_ID",
    "Nommodalidad":"Irrigation_Modality",
    "Idunidadmedida":"Unit_ID",
    "Nomunidad":"Unit",
    "Idcultivo":"Crop_ID",
    "Nomcultivo":"Crop",
    "Sembrada":"Planted_Area_Ha",
    "Cosechada":"Harvested_Area_Ha",
    "Siniestrada":"Damaged_Area_Ha",
    "Volumenproduccion":"Producction_Volume_Ton",
    "Rendimiento":"Yield_Ton_Ha",
    "Preciomediorural":"Avg_Rural_Price_MXN",
    "Valorproduccion":"Producciont_Value_MXN"
}

df=data.rename(columns=columns_tras)


In [4]:
df.head()

,Year,State_ID,State,ID_DDR,DDR_Name,ID_CADER,Name_CADER,Municipality_ID,Municipality_Name,Cycle_ID,...,Unit,Crop_ID,Crop,Planted_Area_Ha,Harvested_Area_Ha,Damaged_Area_Ha,Producction_Volume_Ton,Yield_Ton_Ha,Avg_Rural_Price_MXN,Producciont_Value_MXN
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


In [5]:
df["Irrigation_Modality"].unique()

<ArrowStringArray>
['Riego', 'Temporal']
Length: 2, dtype: str

In [6]:
dfe= df.copy()

dfe["Irrigation_Modality"]= dfe["Irrigation_Modality"].replace({
    "Riego":"Irrigated",
    "Temporal":"Rainfed"
})

dfe["Irrigation_Modality"].unique()

<ArrowStringArray>
['Irrigated', 'Rainfed']
Length: 2, dtype: str

In [7]:
df["Productive_Cycle"].unique()

<ArrowStringArray>
['Otoño-Invierno', 'Primavera-Verano', 'Perennes']
Length: 3, dtype: str

In [8]:
dfe["Productive_Cycle"]= dfe["Productive_Cycle"].replace({
    "Primavera-Verano":"Spring-Summer",
    "Otoño-Invierno":"Atumn-Winter",
    "Perennes":"Perennial"
})

In [9]:
dfe["Productive_Cycle"].unique()

<ArrowStringArray>
['Atumn-Winter', 'Spring-Summer', 'Perennial']
Length: 3, dtype: str

In [10]:
df["Unit"].unique()

<ArrowStringArray>
['Tonelada', 'Metro cuadrado', 'Gruesa', 'Planta', 'Manojo', 'Miles de lts.']
Length: 6, dtype: str

In [11]:
dfe["Unit"]= dfe["Unit"].replace({
    "Tonelada":"Ton",
    "Gruesa":"Gross",
    "Planta":"Plant",
    "Miles de lts":"Thousand_Liters",
    "Manojo":"Bunch",
    "Metro cuadrado":"Square_Meter"
})

dfe["Unit"].unique()

<ArrowStringArray>
['Ton', 'Square_Meter', 'Gross', 'Plant', 'Bunch', 'Miles de lts.']
Length: 6, dtype: str

In [12]:
df["Crop"].unique()

<ArrowStringArray>
[    'Avena forrajera en verde',                      'Brócoli',
                   'Calabacita',                      'Cebolla',
                  'Chile verde',                        'Fresa',
      'Maíz forrajero en verde',                   'Maíz grano',
     'Sorgo forrajero en verde',       'Tomate rojo (jitomate)',
 ...
                      'Quelite',                     'Henequén',
            'Orquídea (planta)',                    'Clyptoria',
                      'Tangelo', 'Ebo (janamargo o veza) grano',
                      'Malanga',                       'Zapupe',
                     'Saramuyo',            'Girasol forrajero']
Length: 314, dtype: str

In [13]:
df["Crop"].value_counts().head(10)

Crop
Maíz grano                  4558
Frijol                      2642
Avena forrajera en verde    1503
Pastos y praderas           1355
Tomate rojo (jitomate)      1145
Chile verde                  861
Sorgo grano                  846
Maíz forrajero en verde      783
Aguacate                     774
Tomate verde                 759
Name: count, dtype: int64

In [14]:
top_crops_en={
    "Maíz grano":"Grain Corn",
    "Frijol":"Bean",
    "Avena forrajera en verde":"Green Forage Oats",
    "Pastos y praderas":"Pastures and Grasslands",
    "Tomate rojo (jitomate)":"Red Tomato",
    "Chile verde":"Green Chili",
    "Sorgo grano":"Grain Sorghum",
    "Maíz forrajero en verde":"Green Forage Corn",
    "Aguacate":"Avocado",
    "Tomate verde":"Green Tomatillo"
}

dfe["Crop"]=df["Crop"].replace(top_crops_en)

###### NOTE: The crop column contains 314 unique crops. To keep visualizations and summaries clear and readable, I will translate only the top 10 most frequent crops (by record count) into English, while keeping the remaining crop names in their original Spanish.

In [13]:
df.head()

,Year,State_ID,State,ID_DDR,DDR_Name,ID_CADER,Name_CADER,Municipality_ID,Municipality_Name,Cycle_ID,...,Unit,Crop_ID,Crop,Planted_Area_Ha,Harvested_Area_Ha,Damaged_Area_Ha,Producction_Volume_Ton,Yield_Ton_Ha,Avg_Rural_Price_MXN,Producciont_Value_MXN
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


In [16]:
df.shape

(35902, 24)

In [17]:
df.describe()

,Year,State_ID,ID_DDR,ID_CADER,Municipality_ID,Cycle_ID,Modality_ID,Unit_ID,Crop_ID,Planted_Area_Ha,Harvested_Area_Ha,Damaged_Area_Ha,Producction_Volume_Ton,Yield_Ton_Ha,Avg_Rural_Price_MXN,Producciont_Value_MXN
count,35902.0,35902.000000,35902.000000,35902.000000,35902.000000,35902.000000,35902.000000,35902.000000,3.590200e+04,35902.000000,35902.000000,35902.000000,3.590200e+04,3.548700e+04,35487.000000,3.590200e+04
mean,2025.0,18.297783,98.443457,2.673500,81.681438,2.074202,1.450866,200201.057629,7.392232e+06,552.295624,532.478255,4.987051,1.323772e+04,7.367522e+02,9167.092843,2.262149e+07
std,0.0,7.509698,47.787521,1.946234,110.593300,0.758678,0.497587,0.546181,1.937965e+06,2564.237631,2525.863663,105.600435,3.600310e+05,4.298630e+04,16326.335838,1.129683e+08
min,2025.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,200201.000000,1.000000e+06,0.000000,0.000000,0.000000,0.000000e+00,8.000000e-02,0.650000,0.000000e+00
25%,2025.0,13.000000,63.000000,1.000000,16.000000,1.000000,1.000000,200201.000000,6.060000e+06,10.000000,9.000000,0.000000,6.240500e+01,2.760000e+00,3963.390000,3.957920e+05
50%,2025.0,19.000000,104.000000,2.000000,42.000000,2.000000,1.000000,200201.000000,7.390000e+06,37.000000,35.000000,0.000000,3.399300e+02,9.500000e+00,6012.950000,1.685129e+06
75%,2025.0,23.000000,126.000000,3.000000,98.000000,3.000000,2.000000,200201.000000,8.130000e+06,200.000000,190.000000,0.000000,1.951630e+03,2.200000e+01,9500.000000,8.378430e+06
max,2025.0,32.000000,193.000000,19.000000,570.000000,3.000000,2.000000,200217.000000,1.507000e+07,179125.500000,179125.500000,9933.000000,4.510000e+07,4.700000e+06,650000.000000,4.968507e+09


In [14]:
df.isnull().sum()

Year                        0
State_ID                    0
State                       0
ID_DDR                      0
DDR_Name                    0
ID_CADER                    0
Name_CADER                  0
Municipality_ID             0
Municipality_Name           0
Cycle_ID                    0
Productive_Cycle            0
Modality_ID                 0
Irrigation_Modality         0
Unit_ID                     0
Unit                        0
Crop_ID                     0
Crop                        0
Planted_Area_Ha             0
Harvested_Area_Ha           0
Damaged_Area_Ha             0
Producction_Volume_Ton      0
Yield_Ton_Ha              415
Avg_Rural_Price_MXN       415
Producciont_Value_MXN       0
dtype: int64

In [19]:
df.duplicated().sum()

np.int64(0)

### REFERENCES

Dirección General del Servicio de Información Agroalimentaria y Pesquera (DGSIAP) (2025) Cierre de la Producción Agrícola 2025 [Dataset]. Secretaría de Agricultura y Desarrollo Rural (SADER). Available at: https://nube.agricultura.gob.mx/datosAbiertos/Agricola.php (Accessed: 25 November 2026).